# Watch a paragraph grow
**Camilo Solano · David Ruiz · Juan Otálvaro**

DistilGPT2 continues a paragraph about Maya, a scientist investigating changes in a river. Each click on **Generate next token** adds one token and updates the attention colors and the probabilities of what could come next. A token can be a word, part of a word, or punctuation.

**Each color identifies one of the 12 attention heads.** A token takes the color of the head giving it the highest attention weight. Darker shading means a stronger relative weight. Hover for all 12 weights; the outline marks the newest token.

In [ ]:
#@title Prepare the experiment
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy>=1.24,<3",
    "matplotlib": "matplotlib>=3.7,<4",
    "ipywidgets": "ipywidgets>=8,<9",
    "transformers": "transformers==4.57.1",
    "safetensors": "safetensors>=0.4",
}
missing = [requirement for module, requirement in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if importlib.util.find_spec("torch") is None:
    command = [sys.executable, "-m", "pip", "install", "-q", "torch>=2.6"]
    if sys.platform != "darwin":
        command += ["--index-url", "https://download.pytorch.org/whl/cpu"]
    subprocess.check_call(command)

# Explicitly enable live controls in Colab, even if google.colab was not pre-imported.
try:
    from google.colab import output as colab_output
except ImportError:
    pass
else:
    colab_output.enable_custom_widget_manager()
print("Dependencies ready.")

Dependencies ready.


In [ ]:
#@title Load DistilGPT2
import hashlib
import html
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

np.set_printoptions(precision=3, suppress=True)
TEAL = "#32847e"
ATTENTION_CMAP = LinearSegmentedColormap.from_list("attention_teal", ["#f3f6f4", TEAL])
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})

def show_matrix(name, matrix, labels=None):
    """A compact, readable matrix without a pandas dependency."""
    matrix = np.asarray(matrix)
    rows = []
    for i, row in enumerate(matrix):
        label = html.escape(labels[i]) if labels else str(i + 1)
        values = "".join(f"<td style='padding:5px 12px;text-align:right'>{v:.3f}</td>" for v in row)
        rows.append(f"<tr><th style='text-align:left;padding-right:12px'>{label}</th>{values}</tr>")
    display(HTML(f"<b>{html.escape(name)}</b> &nbsp; shape {matrix.shape}<table>" + "".join(rows) + "</table>"))

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "distilbert/distilgpt2"
MODEL_REVISION = "2290a62682d06624634c1f46a6ad5be0f47f38aa"
torch.set_num_threads(min(4, os.cpu_count() or 1))
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION,
    use_safetensors=True, attn_implementation="eager",
).to("cpu")
model.eval()
print(f"Loaded {MODEL_ID}: {model.config.n_layer} layers, {model.config.n_head} heads per layer")
print(f"Versions: NumPy {np.__version__}, PyTorch {torch.__version__}, Transformers {transformers.__version__}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded distilbert/distilgpt2: 6 layers, 12 heads per layer
Versions: NumPy 2.1.3, PyTorch 2.11.0+cpu, Transformers 5.18.0


## Try the example
Click **Generate next token** to continue the paragraph. **Undo** removes the last token; **Reset** returns to the starting text. Expand **Edit the paragraph** to try your own text.

Choose **Sample** to enable temperature. **Multihead colors** compares all 12 heads on the same paragraph. Select **One head** to inspect an individual head or the average. Change the layer under **Explore attention**.

In [ ]:
#@title Watch a paragraph grow
"""This source is embedded in the delivered notebook; no companion import is needed."""
CASES = {
    "Paragraph": (
        "Maya was a young scientist who lived in a quiet village near a river. "
        "Every morning, she collected water samples and recorded the temperature. "
        "One day, she noticed that the water was warmer than usual and that several fish had moved upstream. "
        "She checked her notes, compared the samples, and called her colleague Daniel. "
        "Together, they walked along the river to look for the source of the change. "
        "When they reached the old bridge, Maya noticed"
    ),
    "Short sentence": "The cat sat on the",
}


class NextTokenExplorer:
    """Generate one genuine GPT token per click and display its input attention."""

    def __init__(self):
        self.history = []
        self.ids = []
        self.initial_ids = []
        self.initial_text = ""
        self.busy = False
        self.pending = None
        self.rng = torch.Generator().manual_seed(7)
        self.case = widgets.ToggleButtons(options=list(CASES), value="Paragraph", description="Example:")
        self.text = widgets.Textarea(value=CASES["Paragraph"], placeholder="Enter a paragraph…",
                                     layout=widgets.Layout(width="100%", height="125px"))
        self.load = widgets.Button(description="Use this text", icon="check", layout=widgets.Layout(width="150px"))
        self.next = widgets.Button(description="Generate next token", button_style="success", icon="step-forward",
                                   layout=widgets.Layout(width="230px", height="44px"))
        self.next.style.button_color = "#237f78"
        self.undo = widgets.Button(description="Undo", icon="undo", layout=widgets.Layout(width="100px", height="44px"))
        self.reset = widgets.Button(description="Reset", icon="refresh", layout=widgets.Layout(width="100px", height="44px"))
        self.mode = widgets.ToggleButtons(options=[("Most likely", "greedy"), ("Sample", "sample")], description="Decoding:")
        self.temperature = widgets.FloatSlider(value=.8, min=.2, max=1.5, step=.1, disabled=True,
                                               description="Temperature:", continuous_update=False,
                                               layout=widgets.Layout(width="430px"))
        self.layer = widgets.IntSlider(value=3, min=1, max=model.config.n_layer, description="Layer:",
                                      continuous_update=False, layout=widgets.Layout(width="350px"))
        self.head = widgets.Dropdown(options=[("Average of all heads", -1)] + [(f"Head {i+1}", i) for i in range(model.config.n_head)],
                                     value=3, description="Head:", layout=widgets.Layout(width="300px"))
        self.attention_view = widgets.ToggleButtons(options=[("Multihead colors", "multi"), ("One head", "single")], value="multi", description="Colors:")
        self.contrast = widgets.FloatSlider(value=.4, min=.2, max=1., step=.1, description="Contrast:",
                                           continuous_update=False, layout=widgets.Layout(width="430px"))
        self.seed = widgets.BoundedIntText(value=7, min=0, max=9999, disabled=True, description="Seed:")
        self.heatmap = widgets.Checkbox(value=False, description="Show attention row chart")
        self.settings = widgets.Accordion(children=[widgets.VBox([
            self.seed,
            widgets.HBox([self.layer, self.head], layout=widgets.Layout(flex_flow="row wrap")),
            self.contrast, self.heatmap,
            widgets.HTML("<small>Temperature and seed affect sampling. Layer, head and contrast only change the view. "
                         "Choose One head to use the head selector and row chart. "
                         "Contrast changes color intensity, never attention weights. Changing the seed restarts the sampling stream.</small>")
        ])], selected_index=None)
        self.settings.set_title(0, "Explore attention: layer, head, highlight contrast and seed")
        self.status = widgets.HTML()
        self.paragraph = widgets.HTML()
        self.details = widgets.HTML()
        self.distribution = widgets.HTML()
        self.chart = widgets.Output()
        self.editor = widgets.Accordion(children=[widgets.VBox([self.text, self.load])], selected_index=None)
        self.editor.set_title(0, "Edit the paragraph or paste your own text")
        self.panel = widgets.VBox([
            widgets.HTML("<div style='padding:8px 0'><h2 style='margin:0;color:#237f78'>Watch a paragraph grow</h2>"
                         "<p style='font-size:16px'>Click once. Generate one token. See where the model attended.</p></div>"),
            self.case, self.editor,
            widgets.HBox([self.next, self.undo, self.reset], layout=widgets.Layout(flex_flow="row wrap", margin="12px 0")),
            self.status, self.attention_view, self.paragraph, self.details, self.mode, self.temperature,
            self.distribution, self.settings, self.chart,
        ], layout=widgets.Layout(width="auto", max_width="100%", min_width="0", padding="16px", border="1px solid #c9d7d3"))
        self.next.on_click(self.generate_one)
        self.undo.on_click(self.undo_one)
        self.reset.on_click(self.reset_all)
        self.load.on_click(self.load_text)
        self.case.observe(self.choose_case, names="value")
        self.mode.observe(self.change_mode, names="value")
        self.seed.observe(self.change_seed, names="value")
        self.temperature.observe(self.render_distribution, names="value")
        for control in [self.layer, self.head, self.contrast, self.heatmap, self.attention_view]:
            control.observe(self.render, names="value")
        self.load_text()

    def choose_case(self, change=None):
        self.text.value = CASES[self.case.value]
        self.load_text()

    def change_mode(self, change=None):
        self.temperature.disabled = self.seed.disabled = self.mode.value == "greedy"
        self.render_distribution()

    def change_seed(self, change=None):
        self.rng.manual_seed(self.seed.value)

    def load_text(self, button=None):
        text = self.text.value.strip()
        if not text:
            self.status.value = "<b>Please enter some text.</b>"
            return
        ids = tokenizer.encode(text)
        if len(text) > 1600 or len(ids) > 192:
            self.status.value = "<b>Please use a shorter paragraph:</b> at most 192 GPT tokens and 1600 characters."
            return
        self.initial_text, self.initial_ids = text, list(ids)
        self.reset_all()

    def reset_all(self, button=None):
        self.ids, self.history = list(self.initial_ids), []
        self.pending = None
        self.rng.manual_seed(self.seed.value)
        self.render()

    def undo_one(self, button=None):
        if self.history:
            removed = self.history.pop()
            self.ids = list(removed["context_ids"])
            self.rng.set_state(removed["rng_state"])
            self.pending = None
        self.render()

    def predict_context(self):
        if self.pending is None:
            with torch.inference_mode():
                prediction = model(torch.tensor([self.ids], dtype=torch.long), output_attentions=True, use_cache=False)
                self.pending = {
                    "logits": prediction.logits[0, -1].clone(),
                    "rows": np.stack([layer[0, :, -1, :].cpu().numpy() for layer in prediction.attentions]),
                }
        return self.pending

    def probabilities(self):
        logits = self.predict_context()["logits"]
        return torch.softmax(logits / (self.temperature.value if self.mode.value == "sample" else 1.), dim=-1)

    def render_distribution(self, change=None):
        if not self.ids:
            return
        probabilities = self.probabilities()
        values, ids = torch.topk(probabilities, 8)
        rows = []
        for rank, (token_id, value) in enumerate(zip(ids.tolist(), values.tolist())):
            token = html.escape(repr(tokenizer.decode([token_id])))
            rows.append(f"<tr><td style='padding:6px 10px;min-width:110px'><code>{token}</code></td>"
                        f"<td style='padding:6px 10px;width:65%'><div style='height:17px;background:#edf2f0'>"
                        f"<div style='height:17px;background:#32847e;width:{value*100:.3f}%'></div></div></td>"
                        f"<td style='padding:6px 10px;text-align:right'>{value:.2%}</td></tr>")
        remainder = max(0., 1. - float(values.sum()))
        mode = "Sampling probabilities" if self.mode.value == "sample" else "Model probabilities (greedy chooses the largest)"
        self.distribution.value = (
            "<div style='margin-top:18px'><h3 style='margin:0'>What could come next?</h3>"
            f"<p>{mode}. Top 8 candidates out of {len(probabilities):,} vocabulary tokens. "
            "Bars use the full 0–100% scale; displayed probabilities are not renormalized.</p>"
            "<table style='width:100%;font-size:15px'><tr><th style='text-align:left'>Token</th><th>Probability</th><th></th></tr>"
            + "".join(rows) + f"</table><p><b>All other tokens together: {remainder:.2%}</b></p></div>"
        )

    def generate_one(self, button=None):
        if self.busy or len(self.history) >= 32 or (self.history and self.ids[-1] == tokenizer.eos_token_id):
            return
        self.busy = True
        for control in [self.next, self.undo, self.reset, self.load, self.case]:
            control.disabled = True
        self.status.value = "<b>Calculating the next token…</b>"
        try:
            context_ids = list(self.ids)
            with torch.inference_mode():
                prediction = self.predict_context()
                probabilities = self.probabilities()
                rng_state = self.rng.get_state().clone()
                token_id = (torch.multinomial(probabilities, 1, generator=self.rng).item()
                            if self.mode.value == "sample" else torch.argmax(probabilities).item())
                candidate_probs, candidate_ids = torch.topk(probabilities, 5)
                # Only the last query is used for this prediction. Keep all layers
                # and heads so view controls do not require another model call.
                attention_rows = prediction["rows"]
            self.history.append({
                "context_ids": context_ids, "id": token_id, "rows": attention_rows,
                "probability": float(probabilities[token_id]),
                "candidates": list(zip(candidate_ids.tolist(), candidate_probs.tolist())),
                "mode": self.mode.value, "rng_state": rng_state,
            })
            self.ids.append(token_id)
            self.pending = None
            self.render()
        except Exception as error:
            self.status.value = "<b>Could not generate:</b> " + html.escape(str(error))
        finally:
            self.busy = False
            for control in [self.next, self.undo, self.reset, self.load, self.case]:
                control.disabled = False
            self.undo.disabled = not self.history
            self.next.disabled = len(self.history) >= 32 or bool(self.history and self.ids[-1] == tokenizer.eos_token_id)

    def render(self, change=None):
        self.undo.disabled = not self.history
        self.next.disabled = self.busy or len(self.history) >= 32 or bool(self.history and self.ids[-1] == tokenizer.eos_token_id)
        layer_rows = self.predict_context()["rows"][self.layer.value - 1]
        multi = self.attention_view.value == "multi"
        self.head.disabled = multi
        self.heatmap.disabled = multi
        if multi and self.heatmap.value:
            self.heatmap.value = False
            return
        weights = layer_rows.mean(axis=0) if self.head.value == -1 else layer_rows[self.head.value]
        words = [tokenizer.decode([i]) for i in self.ids]
        # A categorical hue identifies the winning head among all heads.
        # Actual per-head weights remain unchanged and are available on hover.
        compared_heads = list(range(model.config.n_head))
        colors = [(37, 99, 235), (234, 88, 12), (22, 163, 74), (147, 51, 234),
                  (220, 38, 38), (8, 145, 178), (219, 39, 119), (101, 120, 22),
                  (67, 56, 202), (146, 64, 14), (13, 148, 136), (202, 138, 4)]
        compared = layer_rows[compared_heads]
        winners = np.argmax(compared, axis=0)
        shown_weights = compared.max(axis=0) if multi else weights
        peak = max(float(shown_weights.max()), 1e-12)
        pieces = []
        for index, (word, weight) in enumerate(zip(words, shown_weights)):
            relative = (float(weight) / peak) ** self.contrast.value
            alpha = 0. if float(weight) == 0 else .10 + .55 * relative
            rgb = colors[int(winners[index])] if multi else (18, 137, 123)
            if multi:
                values = "; ".join(f"head {h+1}: {float(layer_rows[h,index]):.5f} ({float(layer_rows[h,index]):.2%})" for h in compared_heads)
                tooltip_text = f"Position {index+1}, token {word!r}; {values}. Hue: head {int(winners[index])+1}, the highest weight among all heads. Ties use the lowest head number."
            else:
                tooltip_text = f"Position {index+1}, token {word!r}, attention {float(weight):.5f} ({float(weight):.2%})"
            tooltip = html.escape(tooltip_text, quote=True)
            underline = "text-decoration:underline;text-underline-offset:5px;" if index == len(words)-1 else ""
            newest = "outline:2px solid #172b28;font-weight:700;" if self.history and index == len(words)-1 else ""
            visible = " [end of text]" if self.ids[index] == tokenizer.eos_token_id else word
            pieces.append(f"<span title='{tooltip}' style='background:rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha:.3f});white-space:pre-wrap;{underline}{newest}'>{html.escape(visible)}</span>")
        if multi:
            legend = " &nbsp; ".join(f"<span style='display:inline-block;border-bottom:4px solid rgb{rgb};padding:3px 6px'>Head {h+1}</span>" for h,rgb in zip(compared_heads,colors))
            legend += "<div style='margin-top:8px;font-size:13px'>Color = head with the highest attention at this token. Darker = stronger relative weight. Hover for all 12 weights.</div>"
        else:
            label = "Average of all heads" if self.head.value == -1 else f"Head {self.head.value+1}"
            legend = f"<b>{label}</b> · darker teal = stronger relative weight. Hover for exact values."
        self.paragraph.value = (f"<div style='font-size:14px;margin:10px 0;color:inherit'>{legend}</div>"
                                "<div style='font-size:21px;line-height:2;padding:18px;background:#f7f6f2;color:#172b28;border-radius:6px'>"
                                + "".join(pieces) + "</div>")
        head_label = "average of all heads" if self.head.value == -1 else f"head {self.head.value+1}"
        if self.history:
            step = self.history[-1]
            new_token = tokenizer.decode([step["id"]])
            self.status.value = (f"<b>Step {len(self.history)}</b> · Last generated: <code>{html.escape(repr(new_token))}</code>"
                                 f" · Its probability when selected: <b>{step['probability']:.1%}</b>")
        else:
            self.status.value = f"<b>Ready.</b> {len(self.ids)} input tokens. Preview the next-token distribution, then click the green button."
        if self.history and self.ids[-1] == tokenizer.eos_token_id:
            self.status.value += " · Model ended the text. Reset or undo to continue."
        elif len(self.history) >= 32:
            self.status.value += " · 32-token presentation limit reached. Reset or undo to continue."
        view_label = "all 12 heads" if multi else head_label
        self.details.value = (
            f"<p><b>Next prediction:</b> layer {self.layer.value}, {view_label}. "
            "Outlined = newest token; underlined = current query. Changing colors or heads does not change the prediction.</p>"
            "<p>These are attention weights, not a complete explanation of the model's choice."
            + (" Colors identify the strongest head at each position; they do not represent the heads' combined output. A head's color appears only where it has the highest weight." if multi else "")
            + "</p>"
        )
        self.render_distribution()
        with self.chart:
            clear_output(wait=self.heatmap.value)
            if self.heatmap.value:
                positions = np.argsort(-weights, kind="stable")[:12]
                fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
                ax.barh(range(len(positions)), weights[positions], color="#32847e")
                ax.set(yticks=range(len(positions)), yticklabels=[f"{i+1}: {words[i]!r}" for i in positions],
                       xlabel="Original attention weight (not renormalized)",
                       title=f"Strongest keys: {weights[positions].sum():.1%} of this row shown")
                ax.invert_yaxis()
                plt.show()
                plt.close(fig)


explorer = NextTokenExplorer()
explorer.generate_one()
display(explorer.panel)

### What to notice
- The colors and probability chart describe the **next prediction** from the text currently shown.
- Hue identifies the strongest head at a position, not a word category or the heads’ combined output. Some head colors may not appear.
- Temperature changes sampling probabilities. Layer, head and contrast change only the view.

### The Python behind one click
The last input position supplies the next-token probabilities. The example below also extracts its attention weights from all 12 heads in layer 3.

In [ ]:
def one_generation_step(text):
    ids = tokenizer(text, return_tensors="pt")["input_ids"]
    with torch.inference_mode():
        result = model(ids, output_attentions=True, use_cache=False)
        # The final input position predicts the next token.
        probabilities = torch.softmax(result.logits[0, -1], dim=-1)
        next_id = probabilities.argmax().item()
        # Third layer, all heads, last query, every input key.
        attention = result.attentions[2][0, :, -1, :].numpy()
    return tokenizer.decode([next_id]), attention, probabilities[next_id].item()

### Optional: the small NumPy example
The expandable panel uses two-dimensional, invented embeddings to illustrate the calculation. Identical inputs give uniform attention; distinct inputs give different weights. The causal mask blocks later positions.

In [ ]:
#@title Optional: NumPy attention
EMBEDDINGS = {"the": [0.2, 0.1], "cat": [1.0, 0.2], "sat": [0.3, 1.0],
              "on": [-0.2, 0.4], "mat": [0.8, -0.5], "dog": [0.7, 0.6]}

def tokenize(text):
    # Whitespace splitting keeps this teaching example easy to inspect.
    tokens = text.strip().split()
    if not 2 <= len(tokens) <= 10:
        raise ValueError("Please enter 2 to 10 words separated by spaces.")
    if any(len(token) > 24 for token in tokens):
        raise ValueError("Keep each word to 24 characters or fewer.")
    return tokens

def embeddings_for(tokens):
    rows = []
    for token in tokens:
        key = token.lower().strip(".,!?;:")
        digest = hashlib.sha256(key.encode("utf-8")).digest()
        fallback = [round(digest[i] / 127.5 - 1, 2) for i in range(2)]
        rows.append(EMBEDDINGS.get(key, fallback))
    return np.array(rows, dtype=float)

sentence = "The cat sat on the mat"

tokens = tokenize(sentence)

labels = [f"{i + 1}: {word}" for i, word in enumerate(tokens)]

X = embeddings_for(tokens)

def projections(mode="identity"):
    if mode == "identity":
        return np.eye(2), np.eye(2), np.eye(2)
    if mode == "mixed":
        return np.array([[1., .5], [-.5, 1.]]), np.array([[.5, -1.], [1., .5]]), np.array([[1., .5], [0., 1.]])
    raise ValueError("Unknown projection mode.")

W_Q, W_K, W_V = projections("identity")

Q = X @ W_Q

K = X @ W_K

V = X @ W_V

query_index = 1

raw_scores = Q @ K.T

scores = raw_scores / np.sqrt(K.shape[1])

def softmax_rows(scores):
    """Convert each row of scores into nonnegative weights summing to one."""
    scores = np.asarray(scores, dtype=float)
    # Subtracting the row maximum prevents overflow without changing softmax.
    exponentials = np.exp(scores - scores.max(axis=1, keepdims=True))
    return exponentials / exponentials.sum(axis=1, keepdims=True)

attention_weights = softmax_rows(scores)

contextual_vectors = attention_weights @ V

contributions = attention_weights[query_index, :, None] * V

def self_attention(X, W_Q, W_K, W_V, *, causal=False, scale=True, temperature=1.0):
    """Return every intermediate matrix for teaching and inspection.

    X: (n_tokens, d_model). W_Q/W_K: (d_model, d_k).
    W_V: (d_model, d_v). Standard attention uses scale=True, temperature=1.
    Causal attention allows a position to attend to itself and earlier positions.
    """
    X, W_Q, W_K, W_V = [np.asarray(a, dtype=float) for a in (X, W_Q, W_K, W_V)]
    if any(a.ndim != 2 or not np.isfinite(a).all() for a in (X, W_Q, W_K, W_V)):
        raise ValueError("Inputs must be finite two-dimensional matrices.")
    if min(X.shape) < 1 or any(w.shape[0] != X.shape[1] or w.shape[1] < 1 for w in (W_Q, W_K, W_V)):
        raise ValueError("Projection shapes must match X's feature dimension.")
    if W_Q.shape[1] != W_K.shape[1]:
        raise ValueError("Queries and keys must have the same dimension.")
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature must be positive and finite.")
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    raw_scores = Q @ K.T
    divisor = np.sqrt(Q.shape[1]) if scale else 1.0
    scores = raw_scores / divisor / temperature
    mask = np.triu(np.ones(scores.shape, dtype=bool), k=1) if causal else np.zeros(scores.shape, dtype=bool)
    masked_scores = np.where(mask, -np.inf, scores)
    weights = softmax_rows(masked_scores)
    output = weights @ V
    return dict(X=X, Q=Q, K=K, V=V, raw_scores=raw_scores, scores=scores,
                masked_scores=masked_scores, mask=mask, weights=weights, output=output)

TOY_CASES = {
    "A1 · Identical inputs / uniform weights": "cat cat cat cat cat cat",
    "A2 · Distinct inputs / selective weights": "The cat sat on the mat",
}

def plot_toy(result, words, query):
    names = [f"{i + 1}: {word}" for i, word in enumerate(words)]
    weights = result["weights"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    image = axes[0].imshow(weights, vmin=0, vmax=1, cmap=ATTENTION_CMAP)
    axes[0].set(xticks=range(len(words)), yticks=range(len(words)),
                xticklabels=names, yticklabels=names, xlabel="Keys (where information comes from)",
                ylabel="Queries (positions being updated)", title="Attention weights A")
    axes[0].tick_params(axis="x", rotation=45)
    for i in range(len(words)):
        for j in range(len(words)):
            text = "masked" if result["mask"][i, j] else f"{weights[i, j]:.2f}"
            axes[0].text(j, i, text, ha="center", va="center", fontsize=8,
                         color="white" if weights[i, j] > .6 else "#222")
    axes[0].add_patch(Rectangle((-.5, query - .5), len(words), 1, fill=False, ec="#222", lw=2))
    fig.colorbar(image, ax=axes[0], shrink=.75, label="Weight (fixed scale 0 to 1)")
    axes[1].bar(range(len(words)), weights[query], color=TEAL)
    axes[1].set(xticks=range(len(words)), xticklabels=names, ylim=(0, 1), ylabel="Weight",
                title=f"Query {query + 1}: {words[query]}")
    axes[1].tick_params(axis="x", rotation=45)
    for j, weight in enumerate(weights[query]):
        axes[1].text(j, weight + .02, f"{weight:.3f}", ha="center", fontsize=9)
    plt.show()
    plt.close(fig)
    print("Selected query:", result["Q"][query])
    print("Selected weights sum to:", round(weights[query].sum(), 8))
    print("Output vector =", np.round(result["output"][query], 4))

toy_state = {"tokens": list(tokens), "X": X.copy(), "updating": False}

toy_case_select = widgets.Dropdown(options=list(TOY_CASES), value=list(TOY_CASES)[1], description="Case:", layout=widgets.Layout(width="550px"))

text_box = widgets.Text(value=sentence, description="Sentence:", layout=widgets.Layout(width="650px"))

apply_sentence = widgets.Button(description="Apply sentence", button_style="info")

reset_toy = widgets.Button(description="Reset")

word_select = widgets.Dropdown(options=[(name, i) for i, name in enumerate(labels)], value=1, description="Query:")

component_1 = widgets.FloatSlider(value=1, min=-2, max=2, step=.1, description="X component 1:", continuous_update=False, style={"description_width": "110px"})

component_2 = widgets.FloatSlider(value=.2, min=-2, max=2, step=.1, description="X component 2:", continuous_update=False, style={"description_width": "110px"})

projection_select = widgets.Dropdown(options=[("Identity projections", "identity"), ("Mixed projections", "mixed")], description="Matrices:")

causal_check = widgets.Checkbox(value=False, description="Causal mask")

scale_check = widgets.Checkbox(value=True, description="Divide by sqrt(d_k)")

attention_temperature = widgets.FloatSlider(value=1, min=.25, max=2, step=.05, description="Attention temperature:", continuous_update=False, style={"description_width": "150px"})

toy_output, matrix_output = widgets.Output(), widgets.Output()

toy_error = widgets.HTML()

matrix_panel = widgets.Accordion(children=[matrix_output], selected_index=None)

matrix_panel.set_title(0, "Inspect X, projection matrices, Q, K, V, scores and outputs")

def refresh_toy(change=None):
    if toy_state["updating"]:
        return
    query = word_select.value
    toy_state["X"][query] = [component_1.value, component_2.value]
    matrices = projections(projection_select.value)
    result = self_attention(toy_state["X"], *matrices, causal=causal_check.value,
                            scale=scale_check.value, temperature=attention_temperature.value)
    with toy_output:
        clear_output(wait=True)
        plot_toy(result, toy_state["tokens"], query)
    with matrix_output:
        clear_output(wait=True)
        for name, matrix in zip(["W_Q", "W_K", "W_V"], matrices):
            show_matrix(name, matrix)
        for name in ["X", "Q", "K", "V", "masked_scores", "output"]:
            show_matrix(name, result[name], [f"{i+1}: {t}" for i, t in enumerate(toy_state["tokens"])])

def select_toy_word(change=None):
    if toy_state["updating"]:
        return
    toy_state["updating"] = True
    component_1.value, component_2.value = toy_state["X"][word_select.value]
    toy_state["updating"] = False
    refresh_toy()

def apply_toy_sentence(button=None):
    try:
        new_tokens = tokenize(text_box.value)
    except ValueError as error:
        toy_error.value = "<b>" + html.escape(str(error)) + "</b>"
        return
    toy_error.value = ""
    toy_state["updating"] = True
    toy_state["tokens"], toy_state["X"] = new_tokens, embeddings_for(new_tokens)
    word_select.options = [(f"{i+1}: {t}", i) for i, t in enumerate(new_tokens)]
    word_select.value = min(1, len(new_tokens) - 1)
    toy_state["updating"] = False
    select_toy_word()

def reset_toy_example(button=None):
    toy_state["updating"] = True
    toy_case_select.value = list(TOY_CASES)[1]
    text_box.value = sentence
    projection_select.value, causal_check.value, scale_check.value = "identity", False, True
    attention_temperature.value = 1.
    toy_state["updating"] = False
    apply_toy_sentence()

def choose_toy_case(change=None):
    if toy_state["updating"]:
        return
    toy_state["updating"] = True
    text_box.value = TOY_CASES[toy_case_select.value]
    projection_select.value, causal_check.value, scale_check.value = "identity", False, True
    attention_temperature.value = 1.
    toy_state["updating"] = False
    apply_toy_sentence()

word_select.observe(select_toy_word, names="value")

toy_case_select.observe(choose_toy_case, names="value")

for control in [component_1, component_2, projection_select, causal_check, scale_check, attention_temperature]:
    control.observe(refresh_toy, names="value")

apply_sentence.on_click(apply_toy_sentence)

reset_toy.on_click(reset_toy_example)

toy_panel = widgets.VBox([widgets.HTML("<h3 style='color:#23756f;margin:4px 0'>One sentence. One attention head.</h3>"),
                      toy_case_select, text_box, widgets.HBox([apply_sentence, reset_toy]), toy_error,
                      widgets.HBox([word_select, projection_select]),
                      widgets.HBox([component_1, component_2]),
                      widgets.HBox([causal_check, scale_check]), attention_temperature,
                      toy_output, matrix_panel],
                      layout=widgets.Layout(border="1px solid #c9d7d3", padding="16px", width="100%"))


word_buttons = widgets.ToggleButtons(options=list(word_select.options), value=word_select.value,
                                      description="Click a word:", style={"description_width": "100px"})
def sync_word_buttons(change):
    word_select.value = change["new"]
def sync_word_dropdown(change):
    word_buttons.options = word_select.options
    word_buttons.value = change["new"]
word_buttons.observe(sync_word_buttons, names="value")
word_select.observe(sync_word_dropdown, names="value")
toy_panel.children = (word_buttons,) + toy_panel.children
toy_accordion = widgets.Accordion(children=[toy_panel], selected_index=None)
toy_accordion.set_title(0, "Optional NumPy playground: click words, move sliders and inspect Q, K, V")
refresh_toy()
display(toy_accordion)

Accordion(children=(VBox(children=(ToggleButtons(description='Click a word:', index=1, options=(('1: The', 0),…

### Checks and sources
The checks verify that attention weights sum to one and that generation uses the preceding context. DistilGPT2 is a text-completion model and may repeat itself or produce awkward continuations.

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2)

In [ ]:
#@title Verify the calculations
test_X = np.array([[1., 0.], [0., 1.], [1., 1.]])
test_attention = self_attention(test_X, np.eye(2), np.eye(2), np.eye(2), causal=True)
np.testing.assert_allclose(test_attention["weights"].sum(axis=1), 1.)
assert np.all(test_attention["weights"][np.triu_indices(3, 1)] == 0)
if explorer.history:
    latest = explorer.history[-1]
    np.testing.assert_allclose(latest["rows"].sum(axis=-1), 1., atol=1e-5)
    assert latest["rows"].shape[-1] == len(latest["context_ids"])
    assert len(explorer.ids) == len(latest["context_ids"]) + 1
np.testing.assert_allclose(explorer.predict_context()["rows"].sum(axis=-1), 1., atol=1e-5)
assert explorer.predict_context()["rows"].shape[-1] == len(explorer.ids)
print("Checks passed: attention normalizes correctly and uses the context before the generated token.")

Checks passed: attention normalizes correctly and uses the context before the generated token.
